# 🟡 Silver Layer - Dimension Tables

## 📌 Objective

Clean and standardize dimension data for analytics.

## 📊 Tables

* Players
* Species
* Maps
* Rods
* Baits

## 🧠 Transformations

* Data type casting
* String normalization (upper, trim, initcap)
* Handling nulls
* Removing duplicates
* Column standardization

## ⚠️ Special Handling

* Common columns (created_at, ingestion_time)
* Avoid column conflicts across tables

## 📥 Source

* Bronze Tables

## 📤 Target

* Silver Tables (`game_analytics.silver.*`)

## 🎯 Goal

Prepare clean dimension tables for joins in fact table (events)


In [0]:
from pyspark.sql import functions as sf

In [0]:
silver_path = "abfss://curated@storageaccgameanalytics.dfs.core.windows.net/silver"

In [0]:
maps = spark.read.table('game_analytics.bronze.maps')
species = spark.read.table('game_analytics.bronze.species')
players = spark.read.table('game_analytics.bronze.players')

rods = spark.read.table('game_analytics.bronze.rods')
baits = spark.read.table('game_analytics.bronze.baits')

rod_capability = spark.read.table('game_analytics.bronze.rod_capability')
bait_capability = spark.read.table('game_analytics.bronze.bait_capability')

In [0]:
maps_silver = maps.select(
    "map_id",
    "map_name",
    "level_requirement",
    "unlock_cost",
    "is_unlocked_default",
    "cost_per_trip",
    "access_type",
    "ingestion_time"
)

maps_silver = (
    maps_silver
    .withColumn('map_name', sf.initcap(col('map_name')))
    .withColumn('level_requirement', sf.col('level_requirement').cast('int'))
    .withColumn('unlock_cost', sf.col('unlock_cost').cast('int'))
    .withColumn('cost_per_trip', sf.col('cost_per_trip').cast('int'))
    .withColumn('access_type', sf.upper(sf.col('access_type')))
    .withColumn('is_unlocked_default', 
        sf.when(sf.col('is_unlocked_default').isin('True', 'true', True), True).otherwise(False))
    .dropDuplicates(["map_id"])
)

maps_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{silver_path}/silver/maps/data')\
    .saveAsTable('game_analytics.silver.maps')

In [0]:
species_silver = (
    species.select(
        "species_id",
        "species_name",
        "species_type",
        "map_id",
        "rarity",
        "min_weight",
        "max_weight",
        "shadow_size",
        "shadow_speed",
        "active_time",
        "season",
        "difficulty",
        "base_price",
        "created_at",
        "ingestion_time"
    )
    .withColumn('species_name', sf.initcap('species_name'))
    .withColumn('species_type', sf.upper('species_type'))
    .withColumn('rarity', sf.upper('rarity'))
    .withColumn('difficulty', sf.upper('difficulty'))
    .withColumn('active_time', sf.upper('active_time'))
    .withColumn('shadow_size', sf.upper('shadow_size'))    

    .withColumn('min_weight', sf.col('min_weight').cast('double'))
    .withColumn('max_weight', sf.col('max_weight').cast('double'))
    .withColumn('base_price', sf.col('base_price').cast('int'))
    .withColumn('created_at', sf.col('created_at').cast('timestamp'))

    .withColumn(
        'shadow_speed',
        sf.when(sf.lower('shadow_speed').startswith('m'), "MEDIUM")
        .when(sf.lower('shadow_speed').startswith('s'), "SMALL")
        .when(sf.lower('shadow_speed').startswith('f'), "FAST")         
    )

    .withColumn("season_temp", sf.split(sf.col('season'), "–"))
    .withColumn(
       "season_cleaned",
        sf.when(sf.lower(sf.col('season')) == 'all', "All Year")
        .when(
            (sf.size(sf.col("season_temp")) == 1) | 
            (sf.lower(sf.trim(sf.element_at(sf.col("season_temp"), 1))) == 
            sf.lower(sf.trim(sf.element_at(sf.col("season_temp"), -1)))),
            sf.initcap(sf.trim(sf.element_at(sf.col("season_temp"), 1)))
        )
        .otherwise(
            sf.concat(
                sf.initcap(sf.trim(sf.element_at(sf.col("season_temp"), 1))),
                sf.lit("-"),
                sf.initcap(sf.trim(sf.element_at(sf.col("season_temp"), 2)))
            )
        )
    )
    .withColumn(
        'weight_range',
        sf.col('max_weight') - sf.col('min_weight')
    )
    .drop('season_temp', 'season')
    .dropDuplicates(["species_id"])
)

species_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{silver_path}/species/data')\
    .saveAsTable('game_analytics.silver.species')

In [0]:
players_silver = (
    players.select(
        "player_id",
        "first_name",
        "last_name",
        "joined_data",
        "level",
        "country",
        "created_at",
        "ingestion_time"
    )
    .withColumn('first_name', sf.initcap(sf.trim('first_name')))
    .withColumn('last_name', sf.initcap(sf.trim('last_name')))
    .withColumn('country', sf.initcap(sf.trim('country')))
    .withColumn('full_name', sf.concat_ws(
        " ",
        sf.col('first_name'),
        sf.col('last_name')
    ))
    .withColumnRenamed('joined_data', 'joined_date')
    .withColumn('joined_date', sf.col('joined_date').cast('timestamp'))
    .withColumn('level', sf.col('level').cast('int'))
    .dropDuplicates(['player_id'])
)

players_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{silver_path}/players/data')\
    .saveAsTable('game_analytics.silver.players')

In [0]:
rods_silver = (
    rods.select(
        'rod_id',
        'rod_name',
        'buying_cost',
        'currency',
        'durability',
        'level_requirement',
        'ingestion_time'
    )
    .withColumn('rod_name', sf.initcap(sf.trim('rod_name')))
    .withColumn('currency', sf.upper(sf.trim('currency')))
    .withColumn('durability', sf.upper(sf.trim('durability')))
    .withColumn('buying_cost', sf.col('buying_cost').cast('int'))
    .withColumn('level_requirement', sf.col('level_requirement').cast('string'))
    .withColumnRenamed('buying_cost', 'rods_buying_cost')
    .withColumnRenamed('currency', 'rods_currency')
    .withColumnRenamed('durability', 'rods_durability')
    .withColumnRenamed('level_requirement', 'rods_level_requirement')
)


baits_silver = (
    baits.select(
        'bait_id',
        'bait_name',
        'buying_cost',
        'currency',
        'quantity',
        'level_requirement',
        'ingestion_time'
    )
    .withColumn('bait_name', sf.initcap(sf.trim('bait_name')))
    .withColumn('currency', sf.upper(sf.trim('currency')))
    .withColumn('quantity', sf.col('quantity').cast('int'))
    .withColumn('buying_cost', sf.col('buying_cost').cast('int'))
    .withColumn('level_requirement', sf.col('level_requirement').cast('string'))
    .withColumnRenamed('buying_cost', 'baits_buying_cost')
    .withColumnRenamed('currency', 'baits_currency')
    .withColumnRenamed('quantity', 'baits_quantity')
    .withColumnRenamed('level_requirement', 'baits_level_requirement')
)

rod_capability_silver = (
    rod_capability.select(
        'rod_id',
        'capability',
        'ingestion_time'
    )
     .withColumn('capability', sf.initcap(sf.col('capability')))
    .withColumnRenamed('capability', 'rods_capability')
)

bait_capability_silver = (
    bait_capability.select(
        'bait_id',
        'capability',
        'ingestion_time'
    )
    .withColumn('capability', sf.initcap(sf.col('capability')))
    .withColumnRenamed('capability', 'rods_capability')
)

rods_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{silver_path}/equipments/rods/data')\
    .saveAsTable('game_analytics.silver.rods')

baits_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{silver_path}/equipments/baits/data')\
    .saveAsTable('game_analytics.silver.baits')


bait_capability_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{silver_path}/equipments/bait_capability/data')\
    .saveAsTable('game_analytics.silver.bait_capability')

rod_capability_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{silver_path}/equipments/rod_capability/data')\
    .saveAsTable('game_analytics.silver.rod_capability')